In [1]:
# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans as TextKMeans

import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load CSV file
df = pd.read_csv("ExoPlanets-data.csv")

# Display first rows
df.head()

,pl_name,hostname,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,pl_controv_flag,pl_orbper,pl_orbpererr1,...,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2
0,11 Com b,11 Com,2,1,Radial Velocity,2007,Xinglong Station,0,323.21000,0.06000,...,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848
1,11 UMi b,11 UMi,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,0,516.21997,3.20000,...,-1.9765,5.01300,0.005,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903
2,14 And b,14 And,1,1,Radial Velocity,2008,Okayama Astrophysical Observatory,0,186.76000,0.11000,...,-0.7140,5.23133,0.023,-0.023,2.331,0.240,-0.240,4.91781,0.002826,-0.002826
3,14 Her b,14 Her,1,2,Radial Velocity,2002,W. M. Keck Observatory,0,1765.03890,1.67709,...,-0.0073,6.61935,0.023,-0.023,4.714,0.016,-0.016,6.38300,0.000351,-0.000351
4,16 Cyg B b,16 Cyg B,3,1,Radial Velocity,1996,Multiple Observatories,0,798.50000,1.00000,...,-0.0111,6.21500,0.016,-0.016,4.651,0.016,-0.016,6.06428,0.000603,-0.000603


In [3]:
# Dataset information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5759 entries, 0 to 5758
Data columns (total 84 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pl_name          5759 non-null   str    
 1   hostname         5759 non-null   str    
 2   sy_snum          5759 non-null   int64  
 3   sy_pnum          5759 non-null   int64  
 4   discoverymethod  5759 non-null   str    
 5   disc_year        5759 non-null   int64  
 6   disc_facility    5759 non-null   str    
 7   pl_controv_flag  5759 non-null   int64  
 8   pl_orbper        5483 non-null   float64
 9   pl_orbpererr1    4982 non-null   float64
 10  pl_orbpererr2    4982 non-null   float64
 11  pl_orbperlim     5483 non-null   float64
 12  pl_orbsmax       5478 non-null   float64
 13  pl_orbsmaxerr1   2866 non-null   float64
 14  pl_orbsmaxerr2   2866 non-null   float64
 15  pl_orbsmaxlim    5479 non-null   float64
 16  pl_rade          5738 non-null   float64
 17  pl_radeerr1      3982 non

In [4]:
df = df.drop_duplicates()

In [5]:
# Select useful numerical features
features = df.select_dtypes(
    include=['float64','int64']
)

features.head()

,sy_snum,sy_pnum,disc_year,pl_controv_flag,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,pl_orbsmaxerr1,...,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2
0,2,1,2007,0,323.21000,0.06000,-0.05000,0.0,1.178,0.000,...,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848
1,1,1,2009,0,516.21997,3.20000,-3.20000,0.0,1.530,0.070,...,-1.9765,5.01300,0.005,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903
2,1,1,2008,0,186.76000,0.11000,-0.12000,0.0,0.775,0.000,...,-0.7140,5.23133,0.023,-0.023,2.331,0.240,-0.240,4.91781,0.002826,-0.002826
3,1,2,2002,0,1765.03890,1.67709,-1.87256,0.0,2.774,0.109,...,-0.0073,6.61935,0.023,-0.023,4.714,0.016,-0.016,6.38300,0.000351,-0.000351
4,3,1,1996,0,798.50000,1.00000,-1.00000,0.0,1.660,0.030,...,-0.0111,6.21500,0.016,-0.016,4.651,0.016,-0.016,6.06428,0.000603,-0.000603


In [6]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(features)

X_scaled[:5]

array([[ 2.60232813, -0.6656698 , -2.0241983 , -0.07236378, -0.01430422,
        -0.01442719,  0.01546393,  0.01910228, -0.04129209, -0.0241643 ,
         0.03111018,  0.02340609,  1.21892569,         nan,         nan,
         0.02641199,  1.22852449,         nan,         nan,  0.0264143 ,
         2.00275952, -0.21538322,  0.22852794, -0.14587259,  2.00230112,
        -0.21516314,  0.22866291, -0.14587259,  1.07020308, -0.96911188,
         0.89196478, -0.22809753,         nan,         nan,         nan,
                nan,         nan,         nan,         nan,         nan,
        -0.26353782, -0.30745891,         nan,         nan, -0.01346077,
         2.94725108,  1.56043811, -2.49903952,  0.        ,  2.71136311,
         1.84841271, -1.85364293,  0.        , -1.48182711, -0.06196085,
         0.14361187,  0.00794051, -4.39227504, -0.04203292,  0.18335446,
         0.01908661, -0.55030213, -0.0422734 , -0.53328534, -0.23990544,
         0.22424543, -2.55358843, -0.55994568,  0.3

In [8]:
df.head()

,pl_name,hostname,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,pl_controv_flag,pl_orbper,pl_orbpererr1,...,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2
0,11 Com b,11 Com,2,1,Radial Velocity,2007,Xinglong Station,0,323.21000,0.06000,...,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848
1,11 UMi b,11 UMi,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,0,516.21997,3.20000,...,-1.9765,5.01300,0.005,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903
2,14 And b,14 And,1,1,Radial Velocity,2008,Okayama Astrophysical Observatory,0,186.76000,0.11000,...,-0.7140,5.23133,0.023,-0.023,2.331,0.240,-0.240,4.91781,0.002826,-0.002826
3,14 Her b,14 Her,1,2,Radial Velocity,2002,W. M. Keck Observatory,0,1765.03890,1.67709,...,-0.0073,6.61935,0.023,-0.023,4.714,0.016,-0.016,6.38300,0.000351,-0.000351
4,16 Cyg B b,16 Cyg B,3,1,Radial Velocity,1996,Multiple Observatories,0,798.50000,1.00000,...,-0.0111,6.21500,0.016,-0.016,4.651,0.016,-0.016,6.06428,0.000603,-0.000603


In [9]:
df.to_csv(
    "Exoplanets_clustered_results.csv",
    index=False
)

print("File saved successfully")

File saved successfully
